<img src="../Images/DSC_Logo.png" style="width: 400px;">

# Basics of Handling Qualitative Data in Python
# - Preprocessing

Before we can analyze text data computationally, we usually need to prepare it first. In many areas of data science, this preparation step is called **preprocessing**. In qualitative research, this term is not always commonly used. 

Before we analyze text data, we need to make sure that it is **well structured, and free of unnecessary formatting or markup**.

In this notebook, we will prepare the text data that we collected and organized in the previous notebook.

The notebook is divided into two parts:

### 1. Preprocessing Interview Data
We work with interview transcripts and metadata. In this part, we will:
- load and inspect the metadata table,
- generate participant IDs (PIDs),
- partially pseudonymize the interview transcripts,
- and create a metadata table version with personal data and without personal data.

### 2. Preprocessing Web-Scraped Data
We then work with the text collected from Wikipedia pages. In this part, we will:
- extract the textual content from HTML files,
- clean the text,
- remove elements such as references that are not useful for text analysis,
- and apply basic text preprocessing (convert to lowercase).

# 1. Preprocessing Interview Data
## 1.1 Load Metadata Table (.xlsx) into Python

In this step, we will load the metadata table for our interviews into Python using **Pandas**. The metadata table, like the interviews, is fictitious.

The table contains important information about each interview, such as the participant's name, the interviewer, the date, and other details. By loading it into a **DataFrame**, we can view and work with this data in Python, similar to how we would in Excel.  

We use the `read_excel()` function from Pandas. We also install the `openpyxl` library, which allows Python to read Excel files.

In [ ]:
# Install all required libraries for the workshop
!pip install -q -r ../requirements_A02.txt
print("The libraries 'pandas', 'beautifulsoup4', 'openpyxl' have been installed.")

In [ ]:
# Import the pandas library for working with tables
import pandas as pd

# Path to the metadata Excel file
metadata_file = "../Data/DataA/interviews/metadata/metadata_v01.xlsx"

# Load the metadata into a Pandas DataFrame
df_meta = pd.read_excel(metadata_file)

# Display the first few rows of the table
print("Here is the metadata table for our interviews (scroll right to see everything):")
df_meta.head()

## 1.2 Generate Participant IDs (PIDs) and save as Excel table

In this step, we create **Participant IDs (PIDs)** for each interview.  

- A PID is a short, unique code for each participant (e.g., PID001, PID002, …).  
- Using PIDs allows us to partially **pseudonymize the data**, which means we do not to work with the participants' real names in our analysis.
- We add the PIDs as a **new column** in our metadata table.  

After generating the PIDs, we **save the updated table as a new Excel file**, so it can be used in later steps for preprocessing the interview texts.  

> More information on the difference between pseudonymized data and anonymized data: https://www.edpb.europa.eu/sme-data-protection-guide/faq-frequently-asked-questions/answer/what-difference-between_en

In [ ]:
# Create an empty list to store the PIDs
pids = []

# Loop over each row in the DataFrame
for i in range(len(df_meta)):
    pid = "PID{:03d}".format(i+1)  # Creates PID001, PID002, ...
    pids.append(pid)

# Add the list of PIDs as a new column in the DataFrame
df_meta['PID'] = pids

# Display the first few rows of the table (PID is added as last column)
print("Here is the metadata table with the generated pseudonyms for our interviews (scroll right to see the PIDs):")
df_meta.head()

### Note on File Versioning

In this notebook, you will see file names such as `metadata_v01.xlsx` and `metadata_v02.xlsx`.  
This is a simple way of **versioning files**.

Versioning means keeping track of different stages of a file as it changes over time.  
Instead of overwriting a file, we save a new version with an updated version number (for example `v01`, `v02`, `v03`). This allows us to keep earlier versions and makes it easier to **trace what changes were made during the research process**.

Using version numbers in file names is a common and simple practice in many research workflows, especially when working with data that evolves during preprocessing and analysis.

In [ ]:
# Save the table with PID as a new Excel file
output_file = "../Data/DataA/interviews/metadata/metadata_v02.xlsx"
df_meta.to_excel(output_file, index=False)  # index=False avoids adding a row number column

## 1.3 Create Anonymized Metadata Table (v03)

In this step, we create a version of the metadata table that can be safely shared with researchers in your working group who **should not have access to the personal data**.  

- We start with "metadata-interviews_v02.xlsx", which contains PIDs and personal information.  
- We select only **non-personal columns** such as PID, date, topic, interview mode, notes, and transcription details.  
- The resulting table (v03) **no longer contains names, email addresses, or affiliations**.  
- v03 is saved as a new Excel file for analysis and sharing with collaborators who do not need access to personal data.

In [ ]:
import pandas as pd

# Load the pseudonymized metadata (v02)
v02_file = "../Data/DataA/interviews/metadata/metadata_v02.xlsx"
df_v02 = pd.read_excel(v02_file)

# Select only columns that do NOT contain personal data
# Keep PID and other research-relevant info
non_personal_columns = [
    "PID",
    "date",
    "interview_mode",
    "topic",
    "notes",
    "transcription_method",
    "transcription_system"
]

df_v03 = df_v02[non_personal_columns]

# Save the anonymized version as v03
v03_file = "../Data/DataA/interviews/metadata/metadata_v03.xlsx"
df_v03.to_excel(v03_file, index=False)

print(f"v03 metadata saved to: {v03_file}")

## 1.4 Partially Pseudonymize a Single Interview
### 1.4.1 Replace participant name with PID in a single interview file

In this step, we replace the participant's real name with their **Participant ID (PID)**.  

- We first **read the content** of the interview text file.  
- Then we use Python's `replace()` method to substitute the name with the PID.  
- Finally, we **save the updated text** as a new file, so the original text remains unchanged.  

This example only partially (only participants names) pseudonymizes **one file** to show the concept. Later, we will loop through all interviews and replace all participant names automatically.

> **Note:**
>
> In principle, we could also use the file `metadata_v02.xlsx` for the pseudonymization step. This table already contains both the **participant ID (e.g., PID001)** and the **real name of the interviewee (e.g., "Dr. Emily Roberts")**, which would allow us to automate the replacement process. 

>However, for **didactic reasons**, we perform the replacement manually in this example to first illustrate the basic principle of text substitution. In a real research project, the pseudonymization would be implemented by **linking the interview texts with the metadata table**.


In [ ]:
import os

# Path to the folder containing the interview text files
folder_path = "../Data/DataA/interviews/raw"

# Example: We work with the first interview file
file_name = "interview_01.txt"

# Read the content of the file
with open(os.path.join(folder_path, file_name), "r", encoding="utf-8") as f:
    text = f.read()

# Display the first 300 characters before replacement
print("Before replacement:\n", text[:300])

In [ ]:
# Replace the participant's real name with the PID
# Here we manually specify the name and PID for demonstration
text_pseudonymized = text.replace("Dr. Emily Roberts", "PID001")

# Display the first 300 characters after replacement
print("\nAfter replacement:\n", text_pseudonymized[:300])

# Save the pseudonymized text in a new folder
output_folder = "../Data/DataA/interviews/pseudonymized"

# Create the folder if it doesn't exist yet
os.makedirs(output_folder, exist_ok=True)

# Save the pseudonymized text back to a new file
output_file = os.path.join(output_folder, "interview_01_pseud.txt")
with open(output_file, "w", encoding="utf-8") as f:
    f.write(text_pseudonymized)

print(f"\nFile saved as '{output_file}'")

### 1.4.2 Replace all Participants Name Variants with PID

As you can see from the output of the above cell, only the name 'Dr. Emily Roberts' was replaced, not 'Emily'. In interviews, a participant's name can appear in **different forms**: full name, first name, last name, or with titles. To address this:

- We create a **list of all name variants** we want to replace.  
- We loop over this list and replace each variant with the **PID**.  

In [ ]:
import os

# Path to the folder containing the interview text files
folder_path = "../Data/DataA/interviews/raw"

# Work with the first interview file
file_name = "interview_01.txt"

# Save the pseudonymized text in the new folder
output_folder = "../Data/DataA/interviews/pseudonymized"

# Create the folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Read the content of the file
with open(os.path.join(folder_path, file_name), "r", encoding="utf-8") as f:
    text = f.read()

# Define all name variants that need to be replaced
name_variants = ["Dr. Emily Roberts", "Emily", "Emily Roberts"]

# Replace each variant with the PID
text_pseudonymized = text
for name in name_variants:
    text_pseudonymized = text_pseudonymized.replace(name, "PID001")

# Display the first 300 characters after replacement
print("\nAfter replacement:\n", text_pseudonymized[:300])

# Save the pseudonymized text as a new file
output_file = os.path.join(output_folder, "interview_01_pseud.txt")
with open(output_file, "w", encoding="utf-8") as f:
    f.write(text_pseudonymized)

print(f"\nPseudonymized file saved as '{output_file}'")

## 1.5 Partially Pseudonymize all Interviews

---

### **Exercise 1: Extend the Pseudonymization Dictionary**

Below is the pseudonymization code for all participant names in all interviews.  

- The dictionary for PID001 (Emily) is already provided.  
- Your task is to **add the other participants** (PID002, PID003, PID004) following the same pattern (dictionary).  
- Once completed, run the code to pseudonymize all interviews automatically.  

This way, you can see **how adding entries to the dictionary immediately affects the pseudonymization**.

In [ ]:
import os

# Path to the folder containing the original interview text files
input_folder = "../Data/DataA/interviews/raw"

# Path to the new folder where pseudonymized files will be saved
output_folder = "../Data/DataA/interviews/pseudonymized"

# Create the folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Complete replacements dictionary
replacements = {
    "PID001": ["Dr. Emily Roberts", "Emily", "Emily Roberts"],
    "PID002": 
    "PID003": 
    "PID004": 
    # TODO: Add PID002, PID003, PID004 following the same pattern and using the same characters as for PID001 dictionary entry
}

# Loop through all files in the input folder
for file_name in os.listdir(input_folder):
    
    # Only process text files
    if file_name.endswith(".txt"):
        
        # Read the content of the file
        with open(os.path.join(input_folder, file_name), "r", encoding="utf-8") as f:
            text = f.read()
        
        # Replace all participant names with their PID
        for pid, name_variants in replacements.items():
            for name in name_variants:
                text = text.replace(name, pid)
        
        # Save the pseudonymized text to the new folder
        output_file = os.path.join(output_folder, file_name.replace(".txt", "_pseud.txt"))
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(text)
        
        print(f"Pseudonymized '{file_name}'")

---
### Note on Name Replacement

This method works well for exact matches, but it has a limitation:  

- If there is a **spelling mistake** or a **typo** in the transcript, that instance of the name will **not be replaced** correctly.  
- In real-world qualitative data, names might appear in many forms, and manual replacement or exact string matching might miss some occurrences.
- The example shown here is only for pseudonymizing the real names of interview participants. However, there is a lot of other personal data (email addresses, names of other people, locations, etc.) in interview data. For each of these cases, you can write a script or use the “QualiAnon” software for support. The QualiAnon software supports researchers in the partially automated pseudonymization of interviews (https://www.qualiservice.org/en/the-helpdesk/tools.html). 

The demonstration here was intended to show computational thinking using a real use case from qualitative research. In Python, it has the advantage that once a script for pseudonymization has been written, the PIDs can be flexibly exchanged and adjusted. **In addition, the pseudonymization process remains transparent and reproducible throughout the entire research project.**

# 2. Preprocessing Web Scraped Data

In the previous notebook, we downloaded several Wikipedia pages and saved them as **raw HTML files**. These files contain the full structure of the webpage, including HTML tags, links, navigation elements, and other formatting information.

Before we can analyze the text, we first need to **extract the readable content** from these HTML documents.

In this section, we will:
- Load the raw HTML files we scraped earlier
- Use the **BeautifulSoup** library to parse the HTML
- Extract the main text from the webpage
- Clean and organize the text for further analysis

This step is called **preprocessing** because we prepare the raw data so it can be used for text analysis in the next notebook.

## 2.1 Load the raw HTML files

In [ ]:
# Import the library for working with file paths
import os

# Define the folder containing the raw HTML files
folder_path = "../Data/DataA/web-scraping/raw"

# Loop through the files in the folder
for file_name in os.listdir(folder_path):
    
    # Only open HTML files
    if file_name.endswith(".html"):
        
        file_path = os.path.join(folder_path, file_name)
        
        # Open and read the HTML file
        with open(file_path, "r", encoding="utf-8") as f:
            html_content = f.read()
        
        # Print the file name and a short preview
        print(f"\nPreview of {file_name}:")
        print(html_content[:500])

## 2.2 Parsing HTML Files with BeautifulSoup

In this step, we will focus on **extracting the main text** from a single HTML file we downloaded in the previous notebook.  

Webpages contain a lot of information beyond the readable text, such as navigation menus, links, tables, and formatting. To analyze the content, we need to separate the text from the HTML structure.

We use the Python library **BeautifulSoup** to parse the HTML. BeautifulSoup allows us to navigate the HTML structure and extract the parts we are interested in.  

Here, we focus on **paragraph tags (`<p>`)** because most of the main text of an article is stored in these tags. By extracting text from `<p>` elements, we get the readable content while ignoring other parts like menus or sidebars.

In [ ]:
# Import the library for parsing HTML
from bs4 import BeautifulSoup
import os

# Define the path to the raw HTML file we want to parse
html_file = "../Data/DataA/web-scraping/raw/bremen_state.html"

# Open and read the HTML file
with open(html_file, "r", encoding="utf-8") as f:
    html_content = f.read()

# Parse the HTML with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

# Extract all paragraphs (<p> tags) from the HTML
paragraphs = []

for p in soup.find_all("p"):         # Loop through all <p> elements
    text = p.text.strip()             # Get the text and remove leading/trailing spaces
    if text:                           # Only keep non-empty paragraphs
        paragraphs.append(text)       # Add the paragraph to the list

# Combine all paragraphs into a single text block
bremen_text = " ".join(paragraphs)

# Print the first 500 characters to check the result
print(bremen_text[:500])

In [ ]:
# Define the folder where cleaned text files will be saved
clean_folder = "../Data/DataA/web-scraping/clean"

# Create the folder if it doesn't exist
os.makedirs(clean_folder, exist_ok=True)

# Save the extracted text to a .txt file
output_file = "../Data/DataA/web-scraping/clean/bremen_state.txt"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(bremen_text)

print(f"File saved as '{output_file}'")

---

### **Exercise 2: Extract Text from the Other HTML Files**

In this exercise, you will practice extracting text from a different HTML file using the code you just ran.

1. Look at the code below.  
2. Change the **`html_file` variable** so that it points to the next HTML file you want to extract text from (`bremen_city.html` and `bremerhaven_city.html`).
3. Change the **`output_file` variable** so that the extracted text is saved with the provided name in the `clean` folder (`bremen_city.txt` and `bremerhaven_city.txt`).
4. Run the code.  
5. Check the first 500 characters printed to see that the text was extracted correctly.  
6. Make sure the `.txt` file is created in the `clean` folder.

> **Note:**
>
> You do **not** need to modify the parsing logic, just change the file paths.
> 
> We also recommend that you **do not use file names other than those specified here.** In the next notebook, the code provided will refer precisely to these file names.

In [ ]:
# Define the path to the raw HTML file we want to parse
html_file = "../Data/DataA/web-scraping/raw/bremen_state.html"

# Open and read the HTML file
with open(html_file, "r", encoding="utf-8") as f:
    html_content = f.read()

# Parse the HTML with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

# Extract all paragraphs (<p> tags) from the HTML
paragraphs = []

for p in soup.find_all("p"):         # Loop through all <p> elements
    text = p.text.strip()             # Get the text and remove leading/trailing spaces
    if text:                           # Only keep non-empty paragraphs
        paragraphs.append(text)       # Add the paragraph to the list

# Combine all paragraphs into a single text block
bremen_text = " ".join(paragraphs)

# Print the first 500 characters to check the result
print(bremen_text[:500])

In [ ]:
# Save the extracted text to a .txt file
output_file = "../Data/DataA/web-scraping/clean/bremen_state.txt"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(bremen_text)

print(f"File saved as '{output_file}'")

---

### **Exercise 3: Parse All HTML Files Automatically (Advanced)**

In this exercise, you will modify the code to process **all HTML files** in the `raw` folder automatically.  
You will fill in the missing parts to loop over the files and save each extracted text as a `.txt` file in the `clean` folder.

**Instructions:**

1. Fill in the first blank to get the list of all HTML files in the `raw` folder.  
2. Fill in the second blank to loop over each file in that list.  
3. Keep the parsing and saving logic the same – you only need to complete the **loop and the list variable**.  
4. Run the code and check that each HTML file in the folder has a corresponding `.txt` file in the `clean` folder.  
5. Make sure the first 500 characters of the text are printed for verification (as in section 2.1.).

In [ ]:
from bs4 import BeautifulSoup
import os

# Define the input and output folders
raw_folder = "../Data/DataA/web-scraping/raw/"
clean_folder = "../Data/DataA/web-scraping/clean/"

# Get a list of all files in the raw folder
html_files = ___                      # <-- Fill in: get the list of all HTML files (for a hint see [1] below)

# Loop over each file
for file_name in ____:                # <-- Fill in: choose the correct variable

    # Build the full path to the file
    file_path = os.path.join(raw_folder, file_name)
    
    # Only process real HTML files
    if os.path.isfile(file_path) and file_name.endswith(".html"):
        
        # Open and read the HTML file
        with open(file_path, "r", encoding="utf-8") as f:
            html_files = f.read()
        
        # Parse the HTML with BeautifulSoup
        soup = BeautifulSoup(html_files, "html.parser")
        
        # Extract all <p> paragraphs
        paragraphs = []
        for p in soup.find_all("p"):
            text = p.text.strip()
            if text:
                paragraphs.append(text)
        
        # Combine paragraphs into a single text block
        text_block = " ".join(paragraphs)
        
        # Define the output file path
        output_file = os.path.join(clean_folder, file_name.replace(".html", ".txt"))
        
        # Save the extracted text
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(text_block)
        
        # Print status for each processed file
        print(f"Processed {file_name}")

[1] In the code in Notebook A01 Exercise 1, the built-in Python library `os` was already used to list all files in a folder.

---

## 2.3 Preprocessing with Regular Expressions

Before we start analyzing the text in A03, it's helpful to clean some obvious noise from our Wikipedia texts.  

For example, Wikipedia articles often include references like `[9]` or `[12]`. These are not part of the main content and can interfere with text analysis.  

In Python, we can use **regular expressions (regex)** to remove such patterns easily. Regular expressions are patterns used to search, match, or modify specific parts of text. They allow you to find text that follows a certain rule, such as numbers in brackets like `[9]` or email addresses, and then replace or remove it automatically.

>In real research workflows, it is often better to keep intermediate versions of the data. For simplicity in this workshop, we overwrite the cleaned files in `DataA/webscraping/clean` so that we don't create too many intermediate versions and get confused.

In the following code, regex is used to remove all numbers in square brackets from the file bremen_state.txt. **Take a close look at the code and identify where regex is applied.** More information about regex library: https://docs.python.org/3/library/re.html

In [ ]:
import re
import os

# Path to the cleaned Wikipedia text file
input_file = "../Data/DataA/web-scraping/clean/bremen_state.txt"

# Folder where the preprocessed file will be saved
output_folder = "../Data/DataA/web-scraping/clean"

# Read the text file
with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()

# Remove Wikipedia references such as [1], [12], etc.
text_clean = re.sub(r"\[\d+\]", "", text)

# Show the first 1000 characters after preprocessing
print(text_clean[:1000])

# Save the cleaned text (overwrite the original file)
with open(input_file, "w", encoding="utf-8") as f:
    f.write(text_clean)

print(f"File saved as '{input_file}'")


---

### **Exercise 4: Clean additional .txt files**

Look at the code above, copy it in the empty code block below and modify it so that it also cleans the following files:

1. `bremen_city.txt`
2. `bremerhaven_city.txt`

Hint: You only need to modify **one variable** in the code. This is not about writing a loop, but about changing the variables one after the other.

---

### **Exercise 5: Optional Challenge (Advanced)**

This exercise is optional and intended for participants who already had Python skills before this workshop.

So far, we cleaned one file at a time. In real research workflows, we usually want to process many files automatically.

Your task:
Write a loop that goes through **all `.txt` files** in the folder

`../Data/DataA/web-scraping/clean`

and applies the same regex cleaning step to each file.

Hints:
- Look at previous code blocks in this notebook (e.g. Exercise 3).
- You have already seen how to loop through files in a folder.
- You will likely need:
  - `os.listdir()`
  - a `for` loop
  - `os.path.join()`

**If you are new to Python, feel free to skip this challenge and continue with the next sections.**

---

## 2.4 Additional Text Preprocessing with Python
In this section, we use a simple **built-in Python string method** to prepare the text for later analysis. 

Words like **"Bremen"** and **"bremen"** should usually be treated as the same word in text analysis.

To avoid counting them separately, we convert the entire text to **lowercase**.

In [ ]:
# Load one cleaned text file
file_path = "../Data/DataA/web-scraping/clean/bremen_state.txt"

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

# Convert the text to lowercase
text_lower = text.lower()

# Compare the first 200 characters
print("Original text:\n")
print(text[:200])

print("\nLowercase version:\n")
print(text_lower[:200])

In [ ]:
# Save the lowercase text
with open(file_path, "w", encoding="utf-8") as f:
    f.write(text_lower)

---

### **Exercise 6: Preprocess All Web Texts**

In this exercise, you will loop through all cleaned web texts in the `clean` folder and apply **basic preprocessing**: Convert the texts to **lowercase**

After that, you will **overwrite the existing files** with the preprocessed text.

Your task is to **fill in the missing method calls** (string methods) for converting to lowercase word.

In [ ]:
import os

# Path to the folder containing cleaned web texts
folder = "../Data/DataA/web-scraping/clean"

# Loop through all text files
for file_name in os.listdir(folder):
    
    if file_name.endswith(".txt"):
        
        # Read the file
        file_path = os.path.join(folder, file_name)
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
        
        # Convert text to lowercase
        text_lower = text.___            # <-- Fill in: choose the correct string method
        
        # Overwrite the original file
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(text_lower)

        print(f"Overwritten preprocessed file: {file_path}")

---